In [1]:
import pandas as pd

sales = pd.read_csv("indian_sales_practice.csv")
managers = pd.read_csv("city_managers.csv")

print(sales.shape)
print(managers.shape)

(300, 13)
(9, 3)


In [2]:
# INNER merge: only cities found in both tables
inner = pd.merge(sales, managers, on="city", how="inner")
print("inner:", inner.shape)

# LEFT merge: all sales rows, even if the city has no manager
left = pd.merge(sales, managers, on="city", how="left")
print("left:", left.shape)
print("Sales rows with no manager:", left["manager_name"].isnull().sum())

left[["order_id", "city", "sales", "manager_name"]].head()

inner: (263, 15)
left: (300, 15)
Sales rows with no manager: 37


,order_id,city,sales,manager_name
0,1001,Kolkata,4500.0,Priya Das
1,1002,Kolkata,250.0,Priya Das
2,1003,Hyderabad,1440.0,Imran Khan
3,1004,New Delhi,5700.0,Arjun Singh
4,1005,Bengaluru,17100.0,Anita Rao


In [3]:
right = pd.merge(sales, managers, on="city", how="right")
print("right:", right.shape)

# Managers whose city has no sales (the order columns are empty)
right[right["order_id"].isnull()][["city", "manager_name"]]



right: (265, 15)


,city,manager_name
263,Jaipur,Kavita Sharma
264,Ahmedabad,Nikhil Patel


In [5]:
outer = pd.merge(sales, managers, on="city", how="outer")
print("outer:", outer.shape)
print("Orders with no manager:", outer["manager_name"].isnull().sum())
print("Managers with no orders:", outer["order_id"].isnull().sum())

outer: (302, 15)
Orders with no manager: 37
Managers with no orders: 2


In [6]:
outer = pd.merge(sales, managers, on="city", how="outer")
print("outer:", outer.shape)
print("Orders with no manager:", outer["manager_name"].isnull().sum())
print("Managers with no orders:", outer["order_id"].isnull().sum())

outer: (302, 15)
Orders with no manager: 37
Managers with no orders: 2


In [4]:
outer = pd.merge(sales, managers, on="city", how="outer")
print("outer:", outer.shape)
print("Orders with no manager:", outer["manager_name"].isnull().sum())
print("Managers with no orders:", outer["order_id"].isnull().sum())

outer: (302, 15)
Orders with no manager: 37
Managers with no orders: 2


In [7]:
pivot_cat = pd.pivot_table(
    sales,
    values="sales",
    index="region",
    columns="category",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
).round(0)

pivot_cat

category,Electronics,Furniture,Stationery,Total
region,,,,
East,300175.0,275700.0,18760.0,594635.0
North,538125.0,380850.0,33808.0,952782.0
South,1136650.0,941925.0,48190.0,2126765.0
West,486475.0,387400.0,20188.0,894062.0
Total,2461425.0,1985875.0,120945.0,4568245.0


In [9]:
pivot_prod = pd.pivot_table(
    sales,
    values="sales",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0
).round(0)

pivot_prod

product,Bookshelf,Headphones,Notebook Pack,Office Chair,Pen Set,Power Bank,Printer Paper,Smartphone,Study Table
region,,,,,,,,,
East,101700.0,59875.0,10620.0,52800.0,5575.0,13500.0,2565.0,226800.0,121200.0
North,43650.0,41625.0,7260.0,241200.0,8075.0,29400.0,18472.0,467100.0,96000.0
South,406125.0,135250.0,11625.0,300600.0,13188.0,62700.0,23378.0,938700.0,235200.0
West,171000.0,44125.0,6000.0,18000.0,5412.0,25650.0,8775.0,416700.0,198400.0


In [10]:
wide = pd.pivot_table(
    sales, values="sales", index="region",
    columns="category", aggfunc="sum"
).round(0).reset_index()

wide

category,region,Electronics,Furniture,Stationery
0,East,300175.0,275700.0,18760.0
1,North,538125.0,380850.0,33808.0
2,South,1136650.0,941925.0,48190.0
3,West,486475.0,387400.0,20188.0


In [11]:
long = pd.melt(
    wide,
    id_vars="region",
    var_name="category",
    value_name="total_sales"
)

print(long.shape)
long.head(6)

(12, 3)


,region,category,total_sales
0,East,Electronics,300175.0
1,North,Electronics,538125.0
2,South,Electronics,1136650.0
3,West,Electronics,486475.0
4,East,Furniture,275700.0
5,North,Furniture,380850.0


In [12]:
result = (
    sales
    .query("sales > 10000")
    .groupby("region")
    .agg(orders=("order_id", "count"), total_sales=("sales", "sum"))
    .round(0)
    .sort_values("total_sales", ascending=False)
    .reset_index()
)

result

,region,orders,total_sales
0,South,61,1894675.0
1,North,22,837000.0
2,West,22,787550.0
3,East,18,511950.0
